# 02 H-bond microstates (49) and macrostates (11)

Uses the stored per-frame H-bond counts of the incoming peptide against the growth end, split by region:
$\beta_1$ (16-22) and $\beta_2$ (30-40), native or non-native. The H-bond calculation itself is not
repeated here.

Each region gets one of seven states from fold-specific cut points (SI Tables S-thresholds); the pair gives
one of 7 x 7 = 49 microstates. The 11 macrostates follow from a two-tier rule shared by both folds.

In [1]:
import numpy as np
import pandas as pd
import abeta_tips as T

FOLDS = ["nu_8EZD", "S_7Q4M"]

## Observed maxima (column headers of the threshold tables)

In [2]:
rows = {}
for f in FOLDS:
    n1, nn1, n2, nn2 = T.load_hbond_counts(f)
    rows[f] = {"b1 native": n1.max(), "b1 non-native": nn1.max(),
               "b2 native": n2.max(), "b2 non-native": nn2.max(), "frames": len(n1)}
pd.DataFrame(rows)

,nu_8EZD,S_7Q4M
b1 native,6.0,3.0
b1 non-native,4.0,5.0
b2 native,7.0,8.0
b2 non-native,6.0,6.0
frames,1299392.0,804909.0


## Per-region states and the 49 microstates

In [3]:
micro = {}
for f in FOLDS:
    s1, s2 = T.region_states(f)
    micro[f] = T.microstates(s1, s2)

for f in FOLDS:
    print(f"{f}: microstates visited = {len(np.unique(micro[f]))} of 49")

missing = sorted(set(range(49)) - set(np.unique(micro["nu_8EZD"])))
print("never reached in nu (b1 state, b2 state):", [divmod(m, 7) for m in missing])

nu_8EZD: microstates visited = 46 of 49
S_7Q4M: microstates visited = 49 of 49
never reached in nu (b1 state, b2 state): [(4, 0), (4, 2), (4, 4)]


In [4]:
# frames per microstate, rows = beta1 state, columns = beta2 state
for f in FOLDS:
    c = np.bincount(micro[f], minlength=49).reshape(7, 7)
    print(f)
    print(pd.DataFrame(c, index=[f"b1:{n}" for n in T.REGION_STATES], columns=range(7)), "\n")

nu_8EZD
                        0       1      2       3     4      5       6
b1:Melted           79795  130972   5346   85522  1402  77325  107352
b1:Small/Small       8351   36839   2107   32474   888  17811   25721
b1:Small/Large        236     944     95     898    25   1736     626
b1:Large/Small        600    5173    470    4706   175   1405    3433
b1:Large/Large          0      19      0       3     0      2       3
b1:Pure Non-Native  11341   17974   1350   11846   694  12075   11776
b1:Pure Native      17793  191467  11280  195812  4539  38720  140271 

S_7Q4M
                        0      1      2      3    4      5      6
b1:Melted           70336  28326   1205  22740   76  44130  88039
b1:Small/Small       8981  10036   5580  12321  225  25216  24850
b1:Small/Large        451    246    100    156    2   1532    434
b1:Large/Small      12568  38360  25044  51103  984  19777  96056
b1:Large/Large        331    499    297    610   16    391   1242
b1:Pure Non-Native  18768  

## 49 -> 11

Tier 1 maps the seven region states to five tokens (M, $N_{pure}$, $N_{def}$, NN, F); tier 2 is the
decision tree over the ($\beta_1$, $\beta_2$) token pair. The resulting map:

In [5]:
pd.DataFrame(T.MICRO_TO_MACRO.reshape(7, 7),
             index=[f"b1:{n}" for n in T.REGION_STATES],
             columns=[f"b2:{n}" for n in T.REGION_STATES])

,b2:Melted,b2:Small/Small,b2:Small/Large,b2:Large/Small,b2:Large/Large,b2:Pure Non-Native,b2:Pure Native
b1:Melted,0,8,8,3,8,8,3
b1:Small/Small,9,5,5,6,5,5,6
b1:Small/Large,9,5,4,3,5,4,3
b1:Large/Small,2,7,2,10,7,2,10
b1:Large/Large,9,5,5,6,5,5,6
b1:Pure Non-Native,9,5,4,3,5,4,3
b1:Pure Native,2,7,2,10,7,2,1


In [6]:
occ = {}
for f in FOLDS:
    m = T.MICRO_TO_MACRO[micro[f]]
    occ[f] = 100 * np.bincount(m, minlength=11) / len(m)
tab = pd.DataFrame(occ, index=T.MACRO_NAMES).round(2)
tab.loc["b1 alone (b1 Anchored + b1 Exploring)"] = tab.loc["b1 Anchored"] + tab.loc["b1 Exploring"]
tab

,nu_8EZD,S_7Q4M
Melted,6.14,8.74
Pure Native,10.80,6.42
b1 Anchored,5.41,9.89
b2 Anchored,16.78,17.11
Misregistered Trap,1.17,3.68
Frustrated Trap,5.95,6.76
b1 Frustrated,4.48,4.85
b2 Frustrated,15.50,8.06
b2 Exploring,16.55,9.16
b1 Exploring,1.53,3.54
